# 05 — Stage 2 Training (XGBoost Meta-Learner)
Extract latent vectors from trained Stage-1 models, combine with
handcrafted features and regime probabilities, train XGBoost.

**Prerequisites:** NB04 must have completed all 67 folds successfully.

In [ ]:
# Install dependencies
# torch, numpy, pandas are pre-installed by Colab — do NOT reinstall them.
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tqdm pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, pickle, time

# Clone/update repo from GitHub
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2', '__init__.py')):
    raise FileNotFoundError('scalp2 package not found after clone!')

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr

from scalp2.config import load_config
config = load_config(f'{REPO_DIR}/config.yaml')

DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'
CHECKPOINT_DIR = '/content/drive/MyDrive/scalp2/checkpoints'

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'seq_len: {config.model.seq_len}')
print(f'latent_dim: {config.model.fusion.latent_dim}')
print(f'handcrafted_top_k: {config.model.handcrafted_top_k}')

In [ ]:
# Load dataset
df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')
with open(f'{DATA_DIR}/feature_columns.json', 'r') as f:
    feature_cols = json.load(f)

features_array = df[feature_cols].values
labels_array = df['tb_label_cls'].values
returns_array = df['tb_return'].values

print(f'Dataset: {len(df):,} rows, {len(feature_cols)} features')
print(f'Label distribution: {pd.Series(labels_array).value_counts().sort_index().to_dict()}')
print(f'Date range: {df.index[0]} → {df.index[-1]}')

In [ ]:
from scalp2.training.walk_forward import PurgedWalkForwardCV
from scalp2.models.hybrid import HybridEncoder
from scalp2.training.trainer import Stage1Trainer
from scalp2.training.stage2_trainer import Stage2Trainer
from scalp2.regime.hmm import RegimeDetector
from scalp2.utils.serialization import load_fold_artifacts, save_fold_artifacts
from scalp2.utils.metrics import evaluate_predictions

cv = PurgedWalkForwardCV(config.training.walk_forward)
n_folds = cv.n_folds(len(df))
n_features = len(feature_cols)

# Verify checkpoint count matches expected folds
existing_folds = sorted([
    int(p.split('_')[1]) for p in os.listdir(CHECKPOINT_DIR)
    if p.startswith('fold_') and os.path.isdir(os.path.join(CHECKPOINT_DIR, p))
]) if os.path.exists(CHECKPOINT_DIR) else []

print(f'Walk-forward: {n_folds} folds')
print(f'Stage-1 checkpoints found: {len(existing_folds)}')

if len(existing_folds) < n_folds:
    missing = set(range(n_folds)) - set(existing_folds)
    print(f'⚠️ MISSING FOLDS: {sorted(missing)}')
    print('Run NB04 first to complete all folds!')
else:
    print('✅ All Stage-1 checkpoints present')

In [ ]:
# ===== MAIN STAGE 2 TRAINING LOOP =====
stage2 = Stage2Trainer(config, checkpoint_dir=CHECKPOINT_DIR)
all_test_results = []
fold_ics = []
t0 = time.time()

for fold in cv.split(len(df)):
    fold_t0 = time.time()
    print(f'\n{"="*60}')
    print(f'  Fold {fold.fold_idx}/{n_folds-1}')
    print(f'{"="*60}')
    
    # Load Stage-1 artifacts
    try:
        artifacts = load_fold_artifacts(CHECKPOINT_DIR, fold.fold_idx)
    except FileNotFoundError as e:
        print(f'  ❌ Skipping fold {fold.fold_idx}: {e}')
        continue
    
    # Reconstruct model and load weights
    model = HybridEncoder(n_features, config.model)
    model.load_state_dict(artifacts['model_state'])
    
    trainer = Stage1Trainer(model, config.training, checkpoint_dir=CHECKPOINT_DIR)
    regime_detector = artifacts.get('regime_detector')
    
    if regime_detector is None:
        regime_detector = RegimeDetector(config.regime)
        regime_detector.fit(df.iloc[fold.train_start:fold.train_end])
    
    # Scale features
    scaler = artifacts['scaler']
    train_scaled = scaler.transform(features_array[fold.train_start:fold.train_end]).astype(np.float32)
    val_scaled = scaler.transform(features_array[fold.val_start:fold.val_end]).astype(np.float32)
    test_scaled = scaler.transform(features_array[fold.test_start:fold.test_end]).astype(np.float32)
    
    # Run Stage 2
    result = stage2.train_one_fold(
        trainer, regime_detector,
        train_scaled, labels_array[fold.train_start:fold.train_end],
        val_scaled, labels_array[fold.val_start:fold.val_end],
        test_scaled, labels_array[fold.test_start:fold.test_end],
        df.iloc[fold.train_start:fold.train_end],
        df.iloc[fold.val_start:fold.val_end],
        df.iloc[fold.test_start:fold.test_end],
        feature_cols, fold.fold_idx,
    )
    
    # Update fold artifacts with top_feature_indices from Stage 2
    top_indices = result['top_feature_indices']
    save_fold_artifacts(
        CHECKPOINT_DIR, fold.fold_idx,
        artifacts['model_state'], scaler,
        top_indices, feature_cols,
        regime_detector,
        metadata=artifacts.get('metadata'),
    )
    
    # Compute IC for this fold (1-bar and 10-bar)
    seq_len = config.model.seq_len
    test_probs = result['test_probabilities']
    test_labels = result['test_labels']
    n_test = len(test_labels)
    
    # Directional score: P(Long) - P(Short)
    long_score = test_probs[:, 2] - test_probs[:, 0]
    
    # Forward returns for IC computation
    test_returns_1 = returns_array[fold.test_start + seq_len:fold.test_end][:n_test]
    
    if len(test_returns_1) == n_test:
        ic_1, _ = spearmanr(long_score, test_returns_1)
    else:
        ic_1 = np.nan
    
    # 10-bar forward return
    close = df['close'].values
    test_close = close[fold.test_start + seq_len:fold.test_end][:n_test]
    if len(test_close) >= n_test and fold.test_end + 10 <= len(close):
        fwd_10 = close[fold.test_start + seq_len + 10:fold.test_end + 10][:n_test]
        if len(fwd_10) == n_test:
            ret_10 = (fwd_10 - test_close) / test_close
            ic_10, _ = spearmanr(long_score, ret_10)
        else:
            ic_10 = np.nan
    else:
        ic_10 = np.nan
    
    fold_ics.append({'fold': fold.fold_idx, 'ic_1': ic_1, 'ic_10': ic_10})
    
    # Evaluate
    metrics = evaluate_predictions(
        test_probs, test_labels,
        test_returns_1 if len(test_returns_1) == n_test else np.zeros(n_test),
        config.execution.confidence_threshold,
    )
    result['metrics'] = metrics
    all_test_results.append(result)
    
    # Progress log
    elapsed = time.time() - fold_t0
    pred_dist = pd.Series(test_probs.argmax(axis=1)).value_counts(normalize=True).sort_index()
    print(f'  IC(1)={ic_1:+.4f}  IC(10)={ic_10:+.4f}  Acc={metrics.get("accuracy",0):.3f}')
    print(f'  Trades={metrics.get("n_trades",0)}  WinRate={metrics.get("win_rate",0):.3f}  PF={metrics.get("profit_factor",0):.3f}')
    print(f'  Pred=[S:{pred_dist.get(0,0)*100:.1f}% H:{pred_dist.get(1,0)*100:.1f}% L:{pred_dist.get(2,0)*100:.1f}%]')
    print(f'  Time: {elapsed:.1f}s')
    
    # Clean up
    del model, trainer
    torch.cuda.empty_cache()

total_time = time.time() - t0
print(f'\n\n===== Stage 2 complete: {len(all_test_results)} folds in {total_time/60:.1f} min =====')

In [ ]:
# ===== IC ANALYSIS: STAGE 2 (XGBoost) =====
ic_df = pd.DataFrame(fold_ics)
ic_valid_1 = ic_df['ic_1'].dropna()
ic_valid_10 = ic_df['ic_10'].dropna()

print('=' * 70)
print('  STAGE 2 (XGBoost) IC RAPORU')
print('=' * 70)
print(f'Median IC (1-bar):   {ic_valid_1.median():+.4f}')
print(f'Mean IC (1-bar):     {ic_valid_1.mean():+.4f}')
print(f'Median IC (10-bar):  {ic_valid_10.median():+.4f}')
print(f'Mean IC (10-bar):    {ic_valid_10.mean():+.4f}')
print(f'IC > 0 folds (1b):   {(ic_valid_1 > 0).mean()*100:.0f}%')
print(f'IC > 0 folds (10b):  {(ic_valid_10 > 0).mean()*100:.0f}%')

# Aggregate trade metrics
metrics_list = [r['metrics'] for r in all_test_results if r['metrics'].get('n_trades', 0) > 0]
if metrics_list:
    metrics_df = pd.DataFrame(metrics_list)
    print(f'\n--- Aggregate Trade Metrics ---')
    print(f'Total trades:       {metrics_df["n_trades"].sum()}')
    print(f'Mean Win Rate:      {metrics_df["win_rate"].mean():.4f}')
    print(f'Mean Profit Factor: {metrics_df["profit_factor"].mean():.4f}')
    print(f'Mean Sharpe:        {metrics_df["sharpe"].mean():.4f}')
    print(f'Mean Accuracy:      {metrics_df["accuracy"].mean():.4f}')
else:
    print('\n⚠️ No folds produced trades above confidence threshold!')

In [ ]:
# ===== SAVE PREDICTIONS FOR BACKTEST =====
wf_predictions = []
cv_save = PurgedWalkForwardCV(config.training.walk_forward)

for fold, result in zip(cv_save.split(len(df)), all_test_results):
    wf_predictions.append({
        'fold_idx': fold.fold_idx,
        'test_start': fold.test_start,
        'test_end': fold.test_end,
        'test_probabilities': result['test_probabilities'],
        'test_labels': result['test_labels'],
        'regime_probs': result.get('regime_probs'),
    })

save_path = f'{DATA_DIR}/wf_predictions.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(wf_predictions, f)

# Also save Stage-1 only predictions for comparison
# (requires re-extracting Stage-1 probabilities from latents)
print(f'Saved {len(wf_predictions)} fold predictions to {save_path}')
print(f'  regime_probs included: {wf_predictions[0].get("regime_probs") is not None}')
print(f'\n✅ Ready for NB06 (Backtest)')